# Transformer — can rewriting the whole history in activation space change the prediction?

**Hypothesis under test (Sevan, 2026-08-13).** *The reason our edits are failing is because the extra
information in the latent world state, which we're not editing (outside the probe's row space), is
information pertaining to the previous frames / history.*

The companion notebook `gru_history_editing.ipynb` tests this on a GRU, where the history is compressed
into one vector. **This notebook is where the hypothesis can be tested in its strongest form**, because a
causal transformer's carried state *is* the observation history — one slot per frame — and its residual
stream has a **separate representation of every past frame**. If "the un-edited part is the history", here
there is somewhere concrete to write it.

The experiment Sevan specified: train a probe at each layer and window position, then **inject at every
previous frame simultaneously, starting from the earliest layer we can decode from, and repeat the write at
each subsequent layer** (necessary, because the residual stream is recomputed by every block). Then ask
whether the final prediction changes.

> **Structural fact this notebook is built on** (`transformers/transformer_world_state.ipynb`, 2026-08-04):
> a transformer has **two** states and they come apart. The **carried** state is the observation buffer,
> which persists; the **readable** state is the residual stream, which is *recomputed every step*. So an
> activation write is transient **by construction**, not by failure. The `re-applied each step` arm below
> exists precisely so that transience cannot be mistaken for the result.

**Direction brief:** `research/directions/history-editing.md`

In [ ]:
# [1] setup
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

ROOT = Path("/home/sevan/research/physically-implicit-modeling")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "notebooks/experiments/editability/history_editing"))

from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio
from history_tools import ray_centroid, waterfall_grid
from pim.figures.theme import style_ax
from pim.simulator.renderer import render_frame
from pim.simulator.sim import SimConfig
from pim.world_models import load_checkpoint, load_dataset

DEVICE = "cuda"
N_OBJ = 2            # objects in the world
K = 15               # rollout steps scored after the edit
N_EDIT = 256         # held-out edit samples
N_BANK = 1500        # edits sequences used to fit the probe grid
NS = [0, 1, 2, 4, 8, 12]   # history depths: n = PAST frames written beyond the current one

OI = {
    "blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",
    "purple": "#CC79A7", "sky": "#56B4E9", "yellow": "#F0E442", "grey": "#5a5a5a",
}
torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__, "| device", DEVICE, "|", torch.cuda.get_device_name(0))

## Definitions

Metric names and formulas are copied verbatim from the canonical registry `../METRICS_AND_EDITORS.md`
and computed by `scripts/editability_metrics.py`. The §4 metric block is identical to the companion GRU
notebook's, so the two architectures are compared like-for-like.

### Run / data provenance

| name | what it is |
|---|---|
| **Transformer · window 4 · d_model 256** (`runs/transformers/W4`) | Pre-norm causal transformer, `d_model=256` (matched to the GRU's hidden size), `n_layers=4`, `n_heads=4`, banded-causal window 4, 3.23 M params, 300 epochs, seed 0, same next-step MSE objective and same dataset as the GRU — so "recurrence vs attention" is the only architectural variable. Registry row: `../transformers/TRANSFORMER_RUNS.md`. |
| **dataset 4** (`datasets/4_fixed_refl_inview`) | 2 objects, 40 frames, `obs_res = 128`, open boundary, fixed reflectivities, always-in-frustum, radius 0.5, `obs_noise_std = 0.2`, `position_noise_std = 0.04`, `speed_noise_std = direction_noise_std = 0`. |
| **`edits` split** | Held-out sequences with one object teleported at frame `ef = 20`, velocity preserved. |

### Terms

| term | meaning |
|---|---|
| **state span `S`** | `n_layers·(window−1) + 1 = 13` frames — the true size of the carried state, **not** `window`. Pinned by `tests/test_transformer.py::test_buffer_rollout_matches_full_sequence`. This is the history an edit has to overwrite. |
| **residual point ℓ** | a place in the stack where the stream can be read or written: `ℓ = 0` is the encoder port `relu(Linear(obs))`, `ℓ = 1…4` are the block outputs. |
| **window position `j`** | slot in the observation buffer, `j = 0…12`; `j = 12` is the current frame `ef−1` and `j` holds sim frame `ef − 13 + j`. |
| **`n`** (history depth) | past frames written in addition to the current one, so an arm writes positions `12−n … 12`. |
| **δ** (teleport displacement) | `δ = tgt_pos − (pre_pos + dt·v)` for the edited object, `0` for the other. Adding the same `δ` at every position translates the history **rigidly**, preserving velocity. |

### Metrics

| name | formula | units | better | notes |
|---|---|---|---|---|
| **position R² (probe grid)** | `1 − ‖Y − probe(r[ℓ][j])‖²/‖Y − Ȳ‖²`, linear lstsq, scored on **held-out sequences** against the train mean | — | ↑ | one probe per (ℓ, j). Identifies the earliest residual point the position is decodable from — where the multi-layer write starts. |
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)`, `d_· = RMSE(edited₀, gt_·)` over **differing** rays, per sample then averaged | −1…+1 | ↑ | +1 = the edited world, −1 = the unedited world, ≈0 = equidistant **or garbage**. Read against this model's own unsteered row. |
| **Target / Ghost / Collateral RMSE** | `RMSE(edited₀, gt_edited)` over that zone, at rollout step 0 | obs intensity | ↓ | appeared where it should / left where it was / left the other object alone. |
| **GT-traj RMSE** | `mean_s RMSE(edited_s, clean_obs[ef+s])` over the rollout | obs intensity | ↓ | achieved *and held* the true post-edit world. |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)` | ratio | ↓ | > 1 = the edit degraded the rollout. **≈ 1.00 with an unchanged Edit Index means genuine inertness**, which is a different finding from degradation. |
| **probe readout error** | `‖probe(r[ℓ][j]) − target‖` before vs after the write, in sim units | position | ↓ | **the landing diagnostic**. Without it an inert result is ambiguous between "the write failed" and "the write landed and was ignored". |
| **‖Δr‖/‖r‖** | relative size of the activation write, averaged over written sites | ratio | — | how the matched-norm control is built. |

### Editors

| editor | mechanism |
|---|---|
| **Unsteered** | no edit. The reference row. |
| **Activation write · last position only** | pseudoinverse injection at window position 12 alone, at every residual point — the published single-site activation edit, included as the baseline this is meant to improve on. |
| **Activation write · all layers · n** | Sevan's editor: at **every** residual point `ℓ = 0…4` and **every** written position, force the probe readout to `pos(frame) + δ`. Re-applied at each subsequent layer because the stream is recomputed. |
| **Activation write · layer ℓ only / layers ≥ ℓ** | isolates which residual points carry the write, and whether repeating across layers is what matters. |
| **Activation write · re-applied each step** | the same write re-imposed at every rollout step, on the positions that still hold pre-edit frames (their sim frames shift as the buffer advances). Removes "the write washed out" as an explanation. |
| **Random direction, matched norm** | random activation write of the same magnitude — isolates direction from size. |
| **Observation history overwrite · n** | the identical content through the **observation channel**: the last `n+1` buffer frames replaced by *rendered* frames of the translated world. The ceiling. |

In [ ]:
# [2] load the model and the data, and pin the provenance
model, info = load_checkpoint(ROOT / "runs/transformers/W4/best_model.pt", device=DEVICE)
model.eval()
S, L, D = model.state_span, model.cfg.n_layers, model.cfg.d_model

bundle = load_dataset(ROOT / "datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
sim = test.config["dataset"]["sim"]
dt, EF, R = float(sim["dt"]), int(edits.edit_frame), int(sim["obs_res"])

display(Markdown(f"""
| provenance | value |
|---|---|
| model | Transformer · window {model.cfg.window} · d_model {D} (`runs/transformers/W4/best_model.pt`) |
| layers / heads | {L} / {model.cfg.n_heads} |
| **state span `S`** | **{S}** frames = `n_layers·(window−1)+1` — NOT `window` |
| residual points | ℓ = 0 … {L} (0 = encoder port) |
| dataset | `datasets/4_fixed_refl_inview`, `obs_res` = {R} |
| sensing / world noise | {sim['obs_noise_std']} / {sim['position_noise_std']} |
| edit frame `ef` | {EF} |
| probe bank / edit samples | {N_BANK} / {N_EDIT} sequences |
"""))


@torch.no_grad()
def residual_grid(obs_frames, batch=256):
    """(L+1, N, S, D) — the residual stream at every point and position, for window states
    built from `obs_frames` (N, S, R)."""
    outs = []
    for i in range(0, len(obs_frames), batch):
        st = model.state_from_obs(torch.from_numpy(obs_frames[i : i + batch]).float().to(DEVICE))
        outs.append(model.residual_stack(st).cpu().numpy())
    return np.concatenate(outs, axis=1)


def lstsq_fit(X, Y):
    return np.linalg.lstsq(np.c_[X, np.ones(len(X))], Y, rcond=None)[0]


def lstsq_apply(A, X):
    return np.c_[X, np.ones(len(X))] @ A

## §1 — The probe grid: where in the stack, and at which position, is a frame's position readable?

A history write needs to know **where to write**. The residual stream at window position `j` is the
model's representation of the frame in slot `j`, so this fits one linear probe per
(residual point ℓ × window position j) and asks how well it reads that frame's object positions.

Probes are fit on window states anchored at five pre-edit frames, split **by sequence** (80/20), scored
held-out against the train mean. The earliest residual point with usable R² is where the multi-layer
write starts.

In [ ]:
# [3] fit the (residual point x window position) probe grid
ANCHORS = [EF - 4, EF - 3, EF - 2, EF - 1, EF]     # all pre-edit: a state at anchor t holds frames t-S..t-1
pos_b = edits.positions[:N_BANK, :, :N_OBJ, :].reshape(N_BANK, -1, 4).astype(np.float64)
Xg, Yg = [], []
for t in ANCHORS:
    Xg.append(residual_grid(edits.obs[:N_BANK, t - S : t]))
    Yg.append(np.stack([pos_b[:, t - S + j] for j in range(S)], 1))
Xg = np.concatenate(Xg, axis=1)     # (L+1, N*anchors, S, D)
Yg = np.concatenate(Yg, axis=0)     # (N*anchors, S, 4)
n_tr = int(0.8 * Xg.shape[1])
print(f"probe grid: {Xg.shape[1]} window states x {S} positions x {L+1} residual points "
      f"({n_tr} train / {Xg.shape[1]-n_tr} held-out sequences-worth of states)")

grid = np.full((L + 1, S), np.nan)
probes = {}
for ell in range(L + 1):
    for j in range(S):
        Xtr, Xte = Xg[ell, :n_tr, j].astype(np.float64), Xg[ell, n_tr:, j].astype(np.float64)
        Ytr, Yte = Yg[:n_tr, j], Yg[n_tr:, j]
        A = lstsq_fit(Xtr, Ytr)
        grid[ell, j] = 1 - ((lstsq_apply(A, Xte) - Yte) ** 2).sum() / ((Yte - Ytr.mean(0)) ** 2).sum()
        probes[(ell, j)] = (A[:-1].T, A[-1], np.linalg.pinv(A[:-1].T))

hdr = "| residual point | " + " | ".join(f"j={j}" for j in range(S)) + " | mean |"
rows = [hdr, "|" + "---|" * (S + 2)]
for ell in range(L + 1):
    tag = f"ℓ={ell}" + (" (encoder port)" if ell == 0 else "")
    rows.append(f"| {tag} | " + " | ".join(f"{grid[ell, j]:.3f}" for j in range(S))
                + f" | **{grid[ell].mean():.3f}** |")
display(Markdown("**Table 1 — held-out linear position R², by residual point and window position.** "
                 f"Position `j` holds sim frame `ef−{S}+j`; `j={S-1}` is the current frame.\n\n"
                 + "\n".join(rows)))

In [ ]:
# [4] Fig 1 — the probe grid
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.3))
ax = axes[0]
im = ax.imshow(grid, aspect="auto", cmap="viridis", vmin=0.5, vmax=0.85, origin="upper")
ax.set_xticks(range(S))
ax.set_xticklabels([f"{j}" for j in range(S)], fontsize=8)
ax.set_yticks(range(L + 1))
ax.set_yticklabels([f"ℓ={i}" for i in range(L + 1)], fontsize=9)
ax.set_xlabel("window position j  (j=12 = current frame, j=0 = 12 frames ago)")
ax.set_ylabel("residual point")
ax.set_title("(a) position R² — readable at EVERY position, at every layer ≥ 1", fontsize=10)
for ell in range(L + 1):
    for j in range(S):
        ax.text(j, ell, f"{grid[ell, j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if grid[ell, j] < 0.75 else "black")
fig.colorbar(im, ax=ax, label="held-out R²")

ax = axes[1]
style_ax(ax)
for ell, c in zip(range(L + 1), [OI["grey"], OI["sky"], OI["blue"], OI["green"], OI["orange"]]):
    ax.plot(range(S), grid[ell], "o-", ms=4, lw=1.8, color=c,
            label=f"ℓ={ell}" + (" (encoder port)" if ell == 0 else ""))
ax.set_xlabel("window position j  (j=12 = current frame)")
ax.set_ylabel("held-out position R²")
ax.set_title("(b) the profile is flat — the past is as readable as the present", fontsize=10)
ax.legend(fontsize=8, loc="lower right")
ax.set_ylim(0.5, 0.87)

fig.suptitle("Fig 1 — where a frame's object positions are readable in the residual stream "
             "(Transformer · window 4 · d_model 256 · dataset 4)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## §2 — Writing the whole history in activation space

Every written site `(ℓ, j)` gets a pseudoinverse injection driving that probe's readout to
`pos(frame at j) + δ` — the rigidly translated history. Applied at **every** residual point, because the
stream is recomputed by each block, so a single-layer write is erased by the next one.

The arms separate the three things that could each explain an inert result:

* **how far back** the write reaches (`n` sweep),
* **which residual points** carry it (layer-only and `layers ≥ ℓ` sweeps),
* whether the write **washed out** during the rollout (`re-applied each step`),

and two controls decide whether any movement is real: a **matched-norm random** write, and the
**probe readout error** before/after — the landing diagnostic that tells "the write failed" apart from
"the write landed and the model ignored it".

In [ ]:
# [5] the edit setup and the write machinery
NE = N_EDIT
edit_obj = edits.edit_object[:NE].astype(int)
ix = np.arange(NE)
pos_e = edits.positions[:NE, :, :N_OBJ, :].astype(np.float32)
with h5py.File(edits.h5_path, "r") as f:
    vel_e = f["velocities"][:NE, :, :N_OBJ, :].astype(np.float32)

gt_roll = edits.clean_obs[:NE, EF : EF + K].astype(np.float32)
zones = build_edit_zones(
    pre_pos=pos_e[:, EF - 1], tgt_pos=pos_e[:, EF], pre_vel=vel_e[:, EF - 1],
    edit_object=edit_obj, sim=sim, n_obj=N_OBJ,
    traj_pos=pos_e[:, EF : EF + K], gt_edited_traj=gt_roll)

would_be = pos_e[:, EF - 1] + vel_e[:, EF - 1] * dt
delta = np.zeros((NE, N_OBJ, 2), np.float32)
delta[ix, edit_obj] = pos_e[ix, EF, edit_obj] - would_be[ix, edit_obj]

FRAME_OF = {j: EF - S + j for j in range(S)}       # buffer position -> sim frame
state0 = model.state_from_obs(torch.from_numpy(edits.obs[:NE, EF - S : EF]).float().to(DEVICE))
print(f"N = {NE} edits | buffer holds sim frames {FRAME_OF[0]}..{FRAME_OF[S-1]} | "
      f"mean ‖δ‖ = {np.linalg.norm(delta[ix, edit_obj], axis=-1).mean():.3f} sim units")

PROBE_T = {k: tuple(torch.from_numpy(np.ascontiguousarray(a)).float().to(DEVICE) for a in v)
           for k, v in probes.items()}


def targets_for(frames):
    """{position -> (N,4) desired readout} for a buffer whose slot j holds sim frame frames[j]."""
    return {j: torch.from_numpy((pos_e[:, f] + delta).reshape(NE, 4)).float().to(DEVICE)
            for j, f in frames.items()}


TGT0 = targets_for(FRAME_OF)


def make_hook(layers, positions, tgt=None, scale=1.0, randomise=False, seed=0):
    """Pseudoinverse injection at each (residual point, window position) in the write set."""
    tgt = TGT0 if tgt is None else tgt
    gen = torch.Generator(device=DEVICE).manual_seed(seed)

    def hook(i, x):
        if i not in layers:
            return x
        x = x.clone()
        for j in positions:
            if j not in tgt:
                continue
            W, b, W_pinv = PROBE_T[(i, j)]
            step = (tgt[j] - (x[:, j] @ W.T + b)) @ W_pinv.T
            if randomise:   # same magnitude, random direction
                r = torch.randn(step.shape, generator=gen, device=DEVICE)
                step = r * step.norm(dim=1, keepdim=True) / r.norm(dim=1, keepdim=True)
            x[:, j] = x[:, j] + scale * step
        return x

    return hook


ALL_LAYERS = list(range(L + 1))


@torch.no_grad()
def rollout(state, hook=None, steps=K, reapply=False):
    """Free-run; step 0 decodes the edit frame. With `reapply`, the write is re-imposed at every
    step on the positions that still hold pre-edit frames — their slots shift as the buffer advances."""
    pred = model.decode(state, edit=hook) if hook is not None else model.decode(state)
    out, st = [pred], model.advance(state, pred)
    for s in range(1, steps):
        h = None
        if hook is not None and reapply:
            # after s advances, slot j < S-s holds sim frame ef-S+s+j; the rest are generated
            live = {j: EF - S + s + j for j in range(max(S - s, 0))}
            h = make_hook(ALL_LAYERS, list(live), tgt=targets_for(live)) if live else None
        p = model.decode(st, edit=h) if h is not None else model.decode(st)
        out.append(p)
        st = model.advance(st, p)
    return torch.stack(out, 1).cpu().numpy()

In [ ]:
# [6] every arm
ARMS = {"Unsteered": rollout(state0)}
ARMS["Activation write · last position only · all layers"] = rollout(
    state0, make_hook(ALL_LAYERS, [S - 1]))
for n in NS:
    ARMS[f"Activation write · all layers · n={n}"] = rollout(
        state0, make_hook(ALL_LAYERS, list(range(S - 1 - n, S))))
for ell in ALL_LAYERS:
    ARMS[f"Activation write · residual point {ell} only · n={S-1}"] = rollout(
        state0, make_hook([ell], list(range(S))))
for ls in (1, 2, 3):
    ARMS[f"Activation write · residual points >= {ls} · n={S-1}"] = rollout(
        state0, make_hook(list(range(ls, L + 1)), list(range(S))))
ARMS[f"Activation write · all layers · n={S-1} · re-applied each step"] = rollout(
    state0, make_hook(ALL_LAYERS, list(range(S))), reapply=True)
ARMS[f"Random direction, matched norm · all layers · n={S-1}"] = rollout(
    state0, make_hook(ALL_LAYERS, list(range(S)), randomise=True))
for a in (2.0, 4.0):
    ARMS[f"Activation write · all layers · n={S-1} · scaled x{a:g}"] = rollout(
        state0, make_hook(ALL_LAYERS, list(range(S)), scale=a))

# the observation channel: identical content, rendered into the buffer
render_cfg = SimConfig(
    seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
    n_objects=N_OBJ, radius=sim["radius"], n_frames=1, dt=dt, obs_res=R,
    refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
    obs_noise_std=0.0, boundary="open", always_in_frustum=False)
refl = np.linspace(sim["refl_min"], sim["refl_max"], N_OBJ).astype(np.float32)
radii = np.full(N_OBJ, sim["radius"], np.float32)
hist_frames = np.zeros((NE, S, R), np.float32)
for j in range(S):
    p = pos_e[:, FRAME_OF[j]] + delta
    for i in range(NE):
        _, _, inten = render_frame(p[i], radii, refl, render_cfg)
        hist_frames[i, j] = inten
for n in NS:
    buf = edits.obs[:NE, EF - S : EF].astype(np.float32).copy()
    buf[:, S - 1 - n :] = hist_frames[:, S - 1 - n :]
    st = model.state_from_obs(torch.from_numpy(buf).float().to(DEVICE))
    ARMS[f"Observation history overwrite · n={n}"] = rollout(st)

print(f"{len(ARMS)} arms rolled out for K={K} steps each")

In [ ]:
# [7] the LANDING diagnostic — did the write actually take?
@torch.no_grad()
def landing(hook):
    """Probe readout error vs the target, before and after the write, plus the write size."""
    before = model.residual_stack(state0)
    after = model.residual_stack(state0, edit=hook)
    errs_b, errs_a, rel = [], [], []
    for ell in ALL_LAYERS:
        for j in range(S):
            W, b, _ = PROBE_T[(ell, j)]
            for stack, acc in ((before, errs_b), (after, errs_a)):
                read = stack[ell, :, j] @ W.T + b
                acc.append((read - TGT0[j]).norm(dim=1).mean().item())
            rel.append(((after[ell, :, j] - before[ell, :, j]).norm(dim=1)
                        / before[ell, :, j].norm(dim=1)).mean().item())
    return float(np.mean(errs_b)), float(np.mean(errs_a)), float(np.mean(rel))

diag_rows = ["| write set | probe readout error BEFORE (sim units) | AFTER | ‖Δr‖/‖r‖ |",
             "|---|---|---|---|"]
for name, hk in [
    (f"all residual points, all {S} positions", make_hook(ALL_LAYERS, list(range(S)))),
    ("all residual points, last position only", make_hook(ALL_LAYERS, [S - 1])),
    ("residual point 2 only, all positions", make_hook([2], list(range(S)))),
]:
    bfr, aft, rel = landing(hk)
    diag_rows.append(f"| {name} | {bfr:.3f} | **{aft:.3f}** | {rel:.3f} |")
display(Markdown(
    "**Table 2 — the landing diagnostic.** Mean distance between what the probe reads and the target "
    "it was driven to, averaged over all (residual point, window position) sites, before vs after the "
    "write. A small AFTER value means the editor **worked as an editor** — so anything the Edit Index "
    "fails to show is the model ignoring a successful write, not a failed write.\n\n"
    + "\n".join(diag_rows)))

In [ ]:
# [8] the canonical §4 scorecard
cards = {k: edit_scorecard(v, zones, gt_roll) for k, v in ARMS.items()}
for c in cards.values():
    c["fidelity_ratio"] = fidelity_ratio(c, cards["Unsteered"])

rows = ["| arm | channel | **Edit Index** ↑ | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | "
        "GT-traj RMSE ↓ | **fidelity ratio** ↓ |", "|---|---|---|---|---|---|---|---|"]
for k, c in cards.items():
    ch = "observation" if "Observation" in k else ("—" if k == "Unsteered" else "activation")
    flag = " ⚠" if c["fidelity_ratio"] > 1.05 else ""
    rows.append(f"| {k} | {ch} | **{c['edit_index']:+.3f}** | {c['target_rmse']:.3f} | "
                f"{c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} | "
                f"{c['fidelity_ratio']:.2f}{flag} |")
display(Markdown("**Table 3 — the canonical §4 scorecard.** ⚠ marks fidelity ratio > 1.05.\n\n"
                 + "\n".join(rows)))
print(f"\nheadline: activation write all layers n={S-1} "
      f"{cards[f'Activation write · all layers · n={S-1}']['edit_index']:+.3f} "
      f"vs matched-norm RANDOM {cards[f'Random direction, matched norm · all layers · n={S-1}']['edit_index']:+.3f} "
      f"vs observation n={S-1} {cards[f'Observation history overwrite · n={S-1}']['edit_index']:+.3f} "
      f"(unsteered {cards['Unsteered']['edit_index']:+.3f})")

In [ ]:
# [9] Fig 2 — depth of history, depth of stack, and the channel comparison
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
for ax in axes:
    style_ax(ax)

ax = axes[0]
ax.plot(NS, [cards[f"Observation history overwrite · n={n}"]["edit_index"] for n in NS], "o-",
        color=OI["green"], lw=2.4, ms=7, label="observation channel (rendered frames)")
ax.plot(NS, [cards[f"Activation write · all layers · n={n}"]["edit_index"] for n in NS], "s-",
        color=OI["blue"], lw=2.4, ms=7, label="activation channel (all residual points)")
ax.axhline(cards["Unsteered"]["edit_index"], color=OI["grey"], lw=1.6, ls="--", label="unsteered")
ax.axhline(cards[f"Random direction, matched norm · all layers · n={S-1}"]["edit_index"],
           color=OI["red"], lw=1.6, ls=":", label="random direction, matched norm")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("n = past frames written (history depth)")
ax.set_ylabel("Edit Index  (+1 = edited world, −1 = unedited)")
ax.set_title("(a) identical content, identical n — only the channel differs", fontsize=10)
ax.legend(fontsize=7.5, loc="center right")
ax.set_ylim(-0.8, 0.8)

ax = axes[1]
single = [cards[f"Activation write · residual point {ell} only · n={S-1}"]["edit_index"] for ell in ALL_LAYERS]
ax.plot(ALL_LAYERS, single, "o-", color=OI["purple"], lw=2.2, ms=7, label="one residual point only")
cum = [cards[f"Activation write · residual points >= {ls} · n={S-1}"]["edit_index"] for ls in (1, 2, 3)]
ax.plot([0] + [1, 2, 3], [cards[f"Activation write · all layers · n={S-1}"]["edit_index"]] + cum,
        "s--", color=OI["blue"], lw=2.2, ms=6, label="all residual points from ℓ upward")
ax.axhline(cards["Unsteered"]["edit_index"], color=OI["grey"], lw=1.6, ls="--", label="unsteered")
ax.set_xlabel("residual point ℓ")
ax.set_ylabel("Edit Index")
ax.set_title("(b) repeating the write up the stack — where it is written barely matters", fontsize=10)
ax.legend(fontsize=8)
# ⚠ this panel is zoomed ~20x relative to (a) to resolve the layer structure at all; say so, or
# the shape reads as a large effect
span = max(single + cum) - cards["Unsteered"]["edit_index"]
ax.text(0.5, 0.03, f"NOTE: axis is zoomed — the whole vertical range spans {span:.3f} index points.\n"
                   f"The observation channel reaches "
                   f"{cards[f'Observation history overwrite · n={S-1}']['edit_index']:+.2f} on this scale.",
        transform=ax.transAxes, ha="center", va="bottom", fontsize=7.5, color=OI["red"])

ax = axes[2]
bar_arms = ["Unsteered", "Activation write · last position only · all layers",
            f"Activation write · all layers · n={S-1}",
            f"Random direction, matched norm · all layers · n={S-1}",
            f"Activation write · all layers · n={S-1} · re-applied each step",
            f"Activation write · all layers · n={S-1} · scaled x4",
            f"Observation history overwrite · n={S-1}"]
short = ["Unsteered", "Activation write\nlast position only", "Activation write\nall layers, n=12",
         "Random direction,\nmatched norm", "Activation write\nn=12, re-applied each step",
         "Activation write\nn=12, scaled x4", "Observation history\noverwrite n=12"]
vals = [cards[a]["edit_index"] for a in bar_arms]
fid = [cards[a]["fidelity_ratio"] for a in bar_arms]
colors = [OI["grey"], OI["sky"], OI["blue"], OI["red"], OI["purple"], OI["orange"], OI["green"]]
yy = np.arange(len(bar_arms))
ax.barh(yy, vals, color=colors, height=0.7)
for i, (v, f_) in enumerate(zip(vals, fid)):
    ax.text(v + 0.03 if v >= 0 else 0.03, i, f"fid {f_:.2f}", va="center", ha="left", fontsize=7.5,
            color=OI["red"] if f_ > 1.05 else OI["grey"])
ax.set_yticks(yy)
ax.set_yticklabels(short, fontsize=7.5)
ax.invert_yaxis()
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("Edit Index")
ax.set_xlim(-0.85, 0.95)
ax.set_title("(c) the controls", fontsize=10)

fig.suptitle("Fig 2 — rewriting the whole history in activation space vs through observations "
             "(Transformer · window 4 · d_model 256 · dataset 4, N=256 held-out edits)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## §3 — Observation space

Required by `CLAUDE.md` for any claim about an effect on the generations, and built through the same
`waterfall_grid(...)` helper as the GRU notebook, so the two architectures' panels are directly
comparable. Gray on dark; a GT column of clean sim observations; six **noisy** pre-edit context frames
above the dashed edit line; below it, **each column is its own free-run from step 0**; solid green =
target, dashed red = ghost. Each column title carries that arm's Edit Index and fidelity ratio.

In [ ]:
# [10] Fig 3 — the waterfall
wf = ["Unsteered", "Activation write · last position only · all layers",
      f"Activation write · all layers · n={S-1}",
      f"Activation write · all layers · n={S-1} · re-applied each step",
      f"Random direction, matched norm · all layers · n={S-1}",
      f"Activation write · all layers · n={S-1} · scaled x4",
      "Observation history overwrite · n=0", f"Observation history overwrite · n={S-1}"]
wf_short = ["Unsteered", "Activation write\nlast position only", "Activation write\nall layers, n=12",
            "Activation write n=12\nre-applied each step", "Random direction,\nmatched norm",
            "Activation write\nn=12, scaled x4", "Observation history\noverwrite n=0",
            "Observation history\noverwrite n=12"]
labels = {a: f"{s}\nEdit Index {cards[a]['edit_index']:+.2f} · fid {cards[a]['fidelity_ratio']:.2f}"
          for a, s in zip(wf, wf_short)}

samples = list(np.argsort(zones.teleport)[::-1][:3])
fig = waterfall_grid(
    rolls={a: ARMS[a] for a in wf},
    ctx=edits.obs[:NE, EF - 6 : EF].astype(np.float32),
    gt_roll=gt_roll,
    tgt_cx=ray_centroid(zones.target),
    ghost_cx=ray_centroid(zones.ghost),
    samples=samples,
    edit_frame=EF,
    leads_by_one=(),
    title=("Fig 3 — observation-space rollouts after rewriting the whole history: activation channel "
           "vs observation channel\n(Transformer · window 4 · d_model 256 · dataset 4; the three "
           "largest teleports; each column is that arm's own free-run)"),
    labels=labels,
)
out_dir = Path("/tmp/history_editing")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "fig3_transformer_waterfall.png", dpi=125, bbox_inches="tight", facecolor="#0a0a14")
plt.show()
print("saved", out_dir / "fig3_transformer_waterfall.png")

## Summary

**What this notebook measures** (invariant): whether rewriting a transformer's *entire represented
history* in activation space — at every window position, at every residual point — changes what it
predicts, when the same content delivered through observations does. Definitions are above; the numbers
below move as models and experiments evolve.

### Current results (updated 2026-08-13) — Transformer · window 4 · d_model 256 · dataset 4

**1. Every past frame's position is readable, at every depth.** The probe grid is essentially **flat
across window position**: mean linear R² 0.591 (ℓ=0, the encoder port), **0.769 / 0.797 / 0.773 / 0.765**
at ℓ=1…4, with the peak mid-stack at **ℓ=2** — matching the published transformer result that readability
peaks mid-stack rather than at the decoder. The only systematic dip is at `j=0`, the position with the
least context (0.716–0.756). So the past is not hidden: there is a decodable representation of every one
of the 13 carried frames, at every layer from ℓ=1 up. The write has somewhere to go.

**2. The write lands perfectly and the model ignores it.** Driving every site to the translated-history
target drives the probe readout error from **3.289 → 0.000 sim units** (mean over all 5 × 13 sites) with a
write of only **‖Δr‖/‖r‖ = 0.102**. Yet the Edit Index moves **−0.667 → −0.631**, at **fidelity 1.00**.
This is the sharpest form of the thread's negative: not a failed editor, not a degraded rollout — a
*perfectly executed* rewrite of the model's entire represented history that the model declines to act on.

**3. None of the obvious explanations survive.**

| candidate explanation | arm | result |
|---|---|---|
| not enough history written | n = 0 → 12 | −0.643 → −0.631, **saturates by n=4** |
| written at the wrong depth | each residual point alone | −0.666 … −0.647 (best at ℓ=2) |
| not repeated up the stack | ℓ ≥ 1, ≥ 2, ≥ 3 | −0.633, −0.643, −0.661 — all ≈ all-layers |
| the write washed out during the rollout | re-applied at **every** step | **−0.631, identical** |
| the write was too small | scaled ×2, ×4 | −0.604, −0.557, still fidelity 1.00 |
| it is real but small | matched-norm **random** write | **−0.661** vs −0.631 |

The matched-norm random control does leave a **small genuine content effect** — 0.036 index points versus
random's 0.006 — but that is ~3% of the distance to the observation channel's result, and it saturates.

**4. The identical content through observations lands the edit.** Replacing the same buffer frames with
*rendered* frames of the same translated world: **+0.285** (n=0), +0.513, +0.591, +0.663, **+0.681** (n=8),
+0.677 (n=12), at fidelity 0.82 → 0.60, with Target RMSE 0.488 → 0.100 and Ghost 0.588 → 0.101 while
collateral stays flat. Same content, same frames, same `n`; only the channel differs.

### Interpretation (mine, not established)

This is the same dissociation the companion GRU notebook finds, on the architecture where the hypothesis
had its best shot. The GRU could be excused on the grounds that its history is compressed into one vector
with no per-frame slots to write. **The transformer has exactly those slots, the probe reads each of them
at R² ≈ 0.8, and the write into all of them succeeds exactly — and it still does not move the world.**

Taken with the GRU result (where the un-edited complement turns out to be explained by *past observations*
at held-out R² ≈ 0.61–0.66, and by past *positions* at ≈ 0.00), the reading is that a frame's
representation is not a *handle* on that frame. The probe finds a linear direction that **correlates** with
position across the data distribution, and writing along it changes what a probe reads without changing
what the network computes downstream — the prediction is driven by observation-shaped content that the
position direction does not span. What the observation channel supplies is that content.

So the hypothesis's premise — the un-edited part is the history — is **right**, and its implication — write
the history and the edit will land — is **wrong**, because "the history" as the model stores it is not
the history as a position probe reads it.

### Owed / scope limits

* One transformer (`W4`, span 13), one seed, one dataset, position only, N=256 edits.
* **Linear probes only.** An MLP probe reads more (the GRU notebook shows the past is stored *nonlinearly*),
  so an MLP-gradient version of this write is the obvious next arm — though the published thread result is
  that MLP-probe steering does not beat linear steering on this axis.
* Longer windows (`W16`, span 61) are untested here; the published finding that window size is irrelevant
  to editability (arms agree within 0.09) suggests it would not change, but it is not measured.
* The observation arm is fed **clean** renders while the model was trained on noisy observations — a mild
  optimistic bias for the channel that already wins.
* `residual_stack` and the callable `edit` hook are new (`pim/world_models/transformer/model.py`); the
  default forward path is asserted **bit-identical** in `tests/test_transformer.py`.